In [1]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
os.makedirs("dataset_outputs/A2AR/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="A2AR/data/a2ar_train_1",
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [3]:
display(dataset.X.shape)
display(dataset.X_ind.shape)
display(dataset.getDF())
display(dataset.X)


(2400, 0)

(0, 0)

,QSPRID,Y,Drug,QSPRID.1,Y_original
QSPRID,,,,,
A2ARDataset_0000,A2ARDataset_0000,True,Cc1cc(C)n(-c2cc(NC(=O)CCN(C)C)nc(-c3ccc(C)o3)n...,A2ARDataset_0000,1
A2ARDataset_0001,A2ARDataset_0001,True,CNC(=O)C12CC1C(n1cnc3c(NCc4cccc(Cl)c4)nc(C#CCC...,A2ARDataset_0003,1
A2ARDataset_0002,A2ARDataset_0002,True,N#Cc1c(-c2ccccc2)cc(-c2ccco2)nc1N,A2ARDataset_0008,1
A2ARDataset_0003,A2ARDataset_0003,True,CCCn1c(=O)c2nc(-c3ccccc3)[nH]c2n(CCCOC)c1=O,A2ARDataset_0009,1
A2ARDataset_0004,A2ARDataset_0004,True,CCNC(=O)C1OC(n2cnc3c(NCC)nc(C#CCCCc4ccccc4)nc3...,A2ARDataset_0012,1
...,...,...,...,...,...
A2ARDataset_2395,A2ARDataset_2395,True,CNc1ncc(C(=O)NCc2ccc(OC)cc2)c2nc(-c3ccco3)nn12,A2ARDataset_4077,1
A2ARDataset_2396,A2ARDataset_2396,True,Nc1nc(-c2ccco2)c2ncn(C(=O)NCCc3ccccc3)c2n1,A2ARDataset_4078,1
A2ARDataset_2397,A2ARDataset_2397,False,Nc1nc(CSc2nnc(N)s2)nc(Nc2ccc(F)cc2)n1,A2ARDataset_4079,0


""
QSPRID
A2ARDataset_0000
A2ARDataset_0001
A2ARDataset_0002
A2ARDataset_0003
A2ARDataset_0004
...
A2ARDataset_2395
A2ARDataset_2396
A2ARDataset_2397


In [4]:
import torch
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset

# Nastavení zařízení (GPU nebo CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Načtení tokenizeru a modelu
tokenizer = RobertaTokenizerFast.from_pretrained("entropy/roberta_zinc_480m", max_len=128)
model = RobertaForMaskedLM.from_pretrained('entropy/roberta_zinc_480m')

# Přenesení modelu na správné zařízení
model.to(device)

# Připravení collatoru pro padding
collator = DataCollatorWithPadding(tokenizer, padding=True, return_tensors='pt')

# Načtení seznamu SMILES (například z nějaké jiné struktury než datasetu)
smiles = dataset.getDF()["Drug"]

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding

smiles_dataset = SMILESDataset(smiles, tokenizer)

# Vytvoření DataLoaderu (dávky po 32)
batch_size = 32
dataloader = DataLoader(smiles_dataset, batch_size=batch_size, collate_fn=collator)

# Zpracování dat po dávkách
model.eval()  # Převede model do evaluačního režimu (bez trénování)
embeddings_list = []  # Uchováme všechny embeddings

with torch.no_grad():  # Nevytvářet gradienty během evaluace
    for batch in dataloader:
        # Zkontroluj tvar batchů
        print("Batch input_ids tvar:", batch['input_ids'].shape)
        print("Batch attention_mask tvar:", batch['attention_mask'].shape)

        # Přenesení všech vstupů na správné zařízení (GPU nebo CPU)
        input_ids = batch['input_ids'].squeeze(1).to(device)  # Squeeze odstraní extra dimenzi
        attention_mask = batch['attention_mask'].squeeze(1).to(device)  # Squeeze odstraní extra dimenzi

        # Modelování výstupů s hidden states
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        # Získání poslední vrstvy hidden states
        full_embeddings = outputs[1][-1]

        # Výpočet průměrného embeddingu pro každý token
        embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
        
        # Uložení embeddings pro tuto dávku
        embeddings_list.append(embeddings)

# Spojení všech embeddings z dávky do jednoho tensoru
all_embeddings = torch.cat(embeddings_list, dim=0)

# Teď můžeš použít `all_embeddings`, což obsahuje embeddings pro všechny SMILES v datasetu


/tmp/ipykernel_2724/2991385132.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch input_ids tvar: torch.Size([32, 1, 128])
Batch attention_mask tvar: torch.Size([32, 1, 128])
Batch inpu

In [5]:
import pandas as pd

# Převod na NumPy pole a pak na DataFrame
df = pd.DataFrame(all_embeddings.cpu().numpy())


In [6]:
dataset.X = pd.concat([dataset.X, df], axis=1)


/tmp/ipykernel_2724/1146750015.py:1: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  dataset.X = pd.concat([dataset.X, df], axis=1)


In [7]:
display(dataset.X.shape)

(4800, 768)

In [8]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [9]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        # Zjistíme embedding dimenzi na prvním SMILES
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("A2AR/data/a2ar_train_1")

X2_all = load_datasets("A2AR/data/a2ar_val_1")

X3_all = load_datasets("A2AR/data/a2ar_test")

transformer = ChemBERTaTransformer()
X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
X_val_emb = transformer.transform(X2_all.df["Drug"])
X_test_emb = transformer.transform(X3_all.df["Drug"])


/tmp/ipykernel_2724/195630644.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


In [11]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
X_train_emb = X_train_emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, X_train_emb], axis = 1)

In [12]:
X2_all.X = X2_all.X.reset_index(drop=True)
X_val_emb = X_val_emb.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, X_val_emb], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
X_test_emb = X_test_emb.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, X_test_emb], axis = 1)

In [13]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [14]:
imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [15]:
pd.DataFrame(X1).columns[pd.DataFrame(X1).isna().any()].tolist()



[]

In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001
0,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,-0.287995,0.475829,-1.886007,-0.019086,1.685174,-1.575181,-1.036012,0.372024,0.183762,-1.731507
1,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,-0.847985,-0.123727,1.436241,0.550994,-0.118027,0.899331,1.255971,0.138340,0.360206,-1.591655
2,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,1.489012,-0.145865,-0.276977,...,0.016308,-0.615823,-0.601442,-1.669977,1.384967,0.730146,1.281295,-1.245225,-1.563946,1.619037
3,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,0.648537,-1.275505,0.558520,0.940265,-0.989957,0.098584,-0.791973,0.670594,-0.313377,0.854122
4,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,-1.294618,0.290985,0.112522,1.052698,-0.938391,-0.994478,-0.979179,1.762197,-0.801921,-0.096545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3460,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,1.489012,-0.145865,-0.276977,...,1.221856,0.039337,-0.345848,-0.257785,-0.808051,-1.189340,-0.115003,1.100244,1.236962,0.473864
3461,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,0.481551,1.128323,-1.079669,-0.538365,-0.174570,0.675272,0.816715,-0.223604,0.418711,-1.436246
3462,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,1.203936,-0.145865,-0.276977,...,1.169177,-0.821740,-0.880680,0.016980,-1.051404,-0.861372,1.060975,0.735636,2.086703,-0.020123
3463,-0.098367,-0.323209,-0.110594,-0.119876,-0.098367,-0.057831,-0.073798,-0.671586,-0.145865,-0.276977,...,1.154101,-1.817191,-1.057004,2.039169,-0.542305,-0.117311,2.184166,-0.026589,-1.009716,1.977249


In [17]:
from sklearn.ensemble import RandomForestClassifier


In [18]:
from sklearn.model_selection import GridSearchCV

In [21]:
parameter_grid_rf = {"n_estimators": [10, 25, 50, 100, 150],
                    "max_features": ["sqrt", "log2"],
                     "criterion":["gini", "entropy", "log_loss"],
                     "max_depth": [None, 3, 5, 15, 30]
                    }
model = RandomForestClassifier(random_state=42)
gridsearch_rf = GridSearchCV(model, parameter_grid_rf,scoring="matthews_corrcoef", verbose=2)

In [22]:
gridsearch_rf.fit(X1, y1)

Fitting 5 folds for each of 150 candidates, totalling 750 fits


/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/sklearn/base.py:1351: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=gini, max_depth=None, max_features=sqrt, n_estimators=10; total time=   0.8s


/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/sklearn/base.py:1351: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=gini, max_depth=None, max_features=sqrt, n_estimators=10; total time=   0.9s


...

[CV] END criterion=log_loss, max_depth=30, max_features=log2, n_estimators=150; total time=   2.2s


/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/sklearn/base.py:1351: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=log_loss, max_depth=30, max_features=log2, n_estimators=150; total time=   2.2s


/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/sklearn/base.py:1351: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


GridSearchCV(estimator=RandomForestClassifier(random_state=42),
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [None, 3, 5, 15, 30],
                         'max_features': ['sqrt', 'log2'],
                         'n_estimators': [10, 25, 50, 100, 150]},
             scoring='matthews_corrcoef', verbose=2)

In [23]:
print(gridsearch_rf.best_score_)
print(gridsearch_rf.best_params_)


0.9800194374967264
{'criterion': 'entropy', 'max_depth': 15, 'max_features': 'log2', 'n_estimators': 100}


In [24]:
print(gridsearch_rf.scorer_)


make_scorer(matthews_corrcoef, response_method='predict')


In [25]:
print("xd")

xd


In [26]:
best_rf = gridsearch_rf.best_estimator_

In [32]:
predict_rf = best_rf.predict(X2)
from sklearn.metrics import matthews_corrcoef
print(matthews_corrcoef(predict_rf, y2))

0.334811069320768


In [ ]:
from catboost import Pool, cv
from catboost import CatBoostClassifier
cat_features = X1.column
train_pool = Pool(data=X1, label=y1, cat_features=cat_features)

params = {
    'iterations': 1000,
    'learning_rate': 0.1,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC'
}


model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    verbose=50  # vypisování průběhu
)

model.fit(X_train, y_train, cat_features=cat_features)

# Predikce
y_pred = model.predict(X_test)

# Hodnocení
print("Accuracy:", accuracy_score(y_test, y_pred))

cv_results = cv(train_pool, params, fold_count=5, verbose=100)